# 🤖 Emotion-Aware Chatbot — Gradio UI
**Features:**
- 💬 Multi-turn conversation with Mistral-7B empathetic replies
- 🧠 Detected emotion + confidence bar per message
- 🔍 SHAP token highlights (which words drove the emotion)
- 📈 Emotion trend chart across the conversation

**Prerequisites:** Run notebooks 03 (training), 04 (SHAP), 05 (Mistral) first.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%pip -q install gradio transformers torch accelerate bitsandbytes shap pandas numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.7 MB/s eta 0:00:00


In [5]:
import re
import json
import torch
import shap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import gradio as gr
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Device: cpu


## 1. Load Both Models

In [6]:
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Emotional Aware Chatbot')
EXP_ROOT           = DRIVE_PROJECT_ROOT / 'experiments' / 'goemotions_roberta_full_pipeline'
PRIMARY_MODEL_DIR  = EXP_ROOT / 'models' / 'best_model'

def has_model_files(p):
    return p.exists() and (p/'config.json').exists() and (
        (p/'model.safetensors').exists() or (p/'pytorch_model.bin').exists())

def find_latest_checkpoint(base):
    if not base.exists(): return None
    ckpts = [p for p in base.glob('checkpoint-*') if p.is_dir() and has_model_files(p)]
    return max(ckpts, key=lambda p: int(re.search(r'checkpoint-(\d+)', p.name).group(1))) if ckpts else None

ROBERTA_DIR = PRIMARY_MODEL_DIR if has_model_files(PRIMARY_MODEL_DIR) else (
    find_latest_checkpoint(EXP_ROOT / 'results' / 'trainer_runs') or
    find_latest_checkpoint(DRIVE_PROJECT_ROOT / 'results' / 'roberta_runs')
)
if ROBERTA_DIR is None:
    raise FileNotFoundError('No trained RoBERTa model found.')

print('Loading RoBERTa from:', ROBERTA_DIR)
roberta_tok   = AutoTokenizer.from_pretrained(ROBERTA_DIR)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_DIR)
roberta_model.to(device).eval()

LABELS   = [roberta_model.config.id2label[i] for i in range(len(roberta_model.config.id2label))]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
print('Labels:', LABELS)
print('RoBERTa loaded ✅')


Loading RoBERTa from: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/models/best_model


Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Labels: ['Anger', 'Fear', 'Joy', 'Love', 'Neutral', 'Sadness']
RoBERTa loaded ✅


In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'

MISTRAL_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_44bit_compute_dtype=torch.float16,
)

print('Loading Mistral-7B 4-bit...')
mistral_tok   = AutoTokenizer.from_pretrained(MISTRAL_MODEL_ID)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
mistral_model.eval()
mistral_pipe = pipeline(
    'text-generation',
    model=mistral_model,
    tokenizer=mistral_tok,
    return_full_text=False,
)
print('Mistral loaded ✅')
if device == 'cuda':
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Loading Mistral-7B 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

## 2. Core Functions

In [ ]:
# ── Emotion colours for UI ────────────────────────────────────────────────────
EMOTION_COLORS = {
    'Anger'  : '#FF4B4B',
    'Fear'   : '#9B59B6',
    'Joy'    : '#F4D03F',
    'Love'   : '#FF6B9D',
    'Neutral': '#95A5A6',
    'Sadness': '#3498DB',
}

EMOTION_SYSTEM_PROMPTS = {
    'Anger'  : 'You are a calm, empathetic assistant. The user is angry or frustrated. Acknowledge their frustration without judgment and gently offer a constructive perspective.',
    'Fear'   : 'You are a reassuring assistant. The user is anxious or afraid. Validate their concern, offer grounding and reassurance. Keep your tone warm and steady.',
    'Joy'    : 'You are an enthusiastic and warm assistant. The user is happy or excited. Match their positive energy and keep the conversation uplifting.',
    'Love'   : 'You are a warm, caring assistant. The user is expressing love or affection. Respond with genuine warmth and kindness.',
    'Neutral': 'You are a helpful, friendly assistant. The user seems calm. Respond clearly and helpfully.',
    'Sadness': 'You are a compassionate assistant. The user is sad or down. Acknowledge their pain with empathy and offer gentle support.',
}

# ── RoBERTa emotion detection ─────────────────────────────────────────────────
def detect_emotion(text: str) -> dict:
    enc = roberta_tok(text, truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.softmax(roberta_model(**enc).logits, dim=-1).squeeze().cpu().numpy()
    idx = int(probs.argmax())
    return {'label': LABELS[idx], 'confidence': float(probs[idx]),
            'scores': {l: float(p) for l, p in zip(LABELS, probs)}}

# ── SHAP token highlights ─────────────────────────────────────────────────────
def get_shap_html(text: str, emotion_label: str) -> str:
    """
    Returns an HTML string where tokens are highlighted based on their
    SHAP contribution to the predicted emotion class.
    Green = pushes toward predicted emotion, Red = pushes away.
    """
    try:
        def predict_proba(texts):
            if isinstance(texts, np.ndarray): texts = texts.tolist()
            texts = [str(t) for t in texts]
            probs_all = []
            with torch.no_grad():
                for i in range(0, len(texts), 16):
                    enc = roberta_tok(texts[i:i+16], truncation=True, max_length=128,
                                      padding=True, return_tensors='pt').to(device)
                    p = torch.softmax(roberta_model(**enc).logits, dim=-1).detach().cpu().numpy()
                    probs_all.append(p)
            return np.vstack(probs_all)

        masker   = shap.maskers.Text(roberta_tok)
        explainer = shap.Explainer(predict_proba, masker, output_names=LABELS)
        sv        = explainer([text], batch_size=4)

        class_idx   = LABEL2ID[emotion_label]
        token_vals  = sv.values[0][:, class_idx]
        token_data  = sv.data[0]

        # Normalise SHAP values to [-1, 1] for colouring
        abs_max = max(abs(token_vals.max()), abs(token_vals.min()), 1e-9)
        norm_vals = token_vals / abs_max

        html = '<div style="font-size:15px;line-height:2.2;font-family:monospace;">'
        for tok, val in zip(token_data, norm_vals):
            tok_str = str(tok)
            if tok_str in {'<s>', '</s>', '<pad>', ''}: continue
            tok_str = tok_str.replace('Ġ', ' ').replace('▁', ' ')
            if val > 0:
                alpha = min(val * 0.8, 0.85)
                bg    = f'rgba(46,204,113,{alpha:.2f})'
            else:
                alpha = min(abs(val) * 0.8, 0.85)
                bg    = f'rgba(231,76,60,{alpha:.2f})'
            html += (
                f'<span title="SHAP: {token_vals[list(token_data).index(tok)]:.4f}" '
                f'style="background:{bg};border-radius:3px;padding:1px 4px;margin:1px;">'
                f'{tok_str}</span>'
            )
        html += '</div>'
        return html
    except Exception as e:
        return f'<p style="color:gray;">SHAP unavailable: {e}</p>'

# ── Confidence bar chart ──────────────────────────────────────────────────────
def make_confidence_chart(scores: dict) -> plt.Figure:
    labels = list(scores.keys())
    values = [scores[l] for l in labels]
    colors = [EMOTION_COLORS.get(l, '#888') for l in labels]

    fig, ax = plt.subplots(figsize=(5, 3))
    bars = ax.barh(labels, values, color=colors, edgecolor='white', height=0.6)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Confidence', fontsize=10)
    ax.set_title('Emotion Scores', fontsize=11, fontweight='bold')
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    return fig

# ── Emotion trend chart ───────────────────────────────────────────────────────
def make_trend_chart(emotion_history: list) -> plt.Figure:
    """
    emotion_history: list of (turn_number, emotion_label, confidence)
    """
    if not emotion_history:
        fig, ax = plt.subplots(figsize=(6, 2))
        ax.text(0.5, 0.5, 'No conversation yet', ha='center', va='center',
                transform=ax.transAxes, color='gray')
        ax.axis('off')
        return fig

    turns      = [e[0] for e in emotion_history]
    emotions   = [e[1] for e in emotion_history]
    confidences = [e[2] for e in emotion_history]
    colors     = [EMOTION_COLORS.get(e, '#888') for e in emotions]

    fig, ax = plt.subplots(figsize=(max(6, len(turns) * 1.2), 3))
    for i, (t, emo, conf, col) in enumerate(zip(turns, emotions, confidences, colors)):
        ax.scatter(t, conf, color=col, s=160, zorder=5)
        ax.annotate(emo, (t, conf), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8, color=col, fontweight='bold')
    if len(turns) > 1:
        ax.plot(turns, confidences, color='#BDC3C7', linewidth=1.2, zorder=1)

    ax.set_ylim(0, 1.15)
    ax.set_xticks(turns)
    ax.set_xticklabels([f'Turn {t}' for t in turns], fontsize=8)
    ax.set_ylabel('Confidence', fontsize=9)
    ax.set_title('Emotion Trend', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    return fig

# ── Mistral response ──────────────────────────────────────────────────────────
def mistral_reply(
    user_text: str,
    emotion: str,
    confidence: float,
    history: list,         # list of [user_msg, bot_msg] pairs
    max_new_tokens: int = 256,
    temperature: float = 0.7,
) -> str:
    system = EMOTION_SYSTEM_PROMPTS.get(emotion, 'You are a helpful, empathetic assistant.')
    prompt = '<s>'
    for user_h, bot_h in (history[-4:] if len(history) > 4 else history):
        prompt += f'[INST] {user_h} [/INST] {bot_h} </s>'
    emotion_ctx = f'[Detected emotion: {emotion} ({confidence:.0%})]\n{user_text}'
    prompt += f'[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{emotion_ctx} [/INST]'

    out = mistral_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=mistral_tok.eos_token_id,
    )
    return out[0]['generated_text'].strip()

print('Core functions ready ✅')


## 3. Gradio App

In [ ]:
# ── State held outside Gradio (mutable across calls) ─────────────────────────
emotion_history = []   # list of (turn, label, confidence)

def chat(
    user_message: str,
    chat_history: list,
    max_tokens: int,
    temperature: float,
    run_shap: bool,
):
    """
    Main Gradio callback.
    Yields updates progressively so the UI feels responsive.
    """
    if not user_message.strip():
        yield chat_history, None, '<p>Please type a message.</p>', None
        return

    # Step 1: Emotion detection (fast)
    emo_result  = detect_emotion(user_message)
    emotion     = emo_result['label']
    confidence  = emo_result['confidence']
    scores      = emo_result['scores']
    color       = EMOTION_COLORS.get(emotion, '#888')

    # Update emotion history
    turn = len(emotion_history) + 1
    emotion_history.append((turn, emotion, confidence))

    conf_fig  = make_confidence_chart(scores)
    trend_fig = make_trend_chart(emotion_history)

    # Yield early so emotion info shows while Mistral generates
    yield (
        chat_history,
        conf_fig,
        f'<p style="color:{color};font-size:13px;">⏳ Generating response for <b>{emotion}</b> emotion...</p>',
        trend_fig,
    )

    # Step 2: Mistral response
    reply = mistral_reply(user_message, emotion, confidence, chat_history, max_tokens, temperature)

    # Update chat history
    chat_history = chat_history + [[user_message, reply]]

    # Step 3: SHAP (optional — slow ~15s per message)
    if run_shap:
        shap_html = get_shap_html(user_message, emotion)
    else:
        shap_html = (
            f'<div style="padding:10px;background:#f8f9fa;border-radius:8px;">'
            f'<b style="color:{color}">{emotion}</b> detected with '
            f'<b>{confidence:.1%}</b> confidence.<br>'
            f'<small style="color:gray;">Enable "Show SHAP" to see token highlights.</small></div>'
        )

    yield chat_history, conf_fig, shap_html, trend_fig


def reset_chat():
    global emotion_history
    emotion_history = []
    return [], None, '<p style="color:gray;">Conversation reset.</p>', make_trend_chart([])


# ── Gradio Layout ─────────────────────────────────────────────────────────────
with gr.Blocks(
    title='Emotion-Aware Chatbot',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate'),
    css='''
        .chat-col   { min-height: 520px; }
        .panel      { background: #f8fafc; border-radius: 12px; padding: 12px; }
        footer      { display: none !important; }
    '''
) as demo:

    gr.Markdown(
        '# 🤖 Emotion-Aware Chatbot\n'
        '**RoBERTa** detects your emotion · **Mistral-7B** crafts an empathetic reply · '
        '**SHAP** shows which words drove the emotion'
    )

    with gr.Row():
        # ── Left column: Chat ──────────────────────────────────────────────
        with gr.Column(scale=3, elem_classes='chat-col'):
            chatbot = gr.Chatbot(
                label='Conversation',
                height=420,
                bubble_full_width=False,
                avatar_images=(None, '🤖'),
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder='Type your message here...',
                    show_label=False,
                    scale=5,
                    lines=2,
                )
                send_btn = gr.Button('Send 💬', variant='primary', scale=1)

            with gr.Row():
                reset_btn = gr.Button('🔄 Reset', variant='secondary', scale=1)
                shap_toggle = gr.Checkbox(
                    label='Show SHAP highlights (slower ~15s)',
                    value=False,
                    scale=2,
                )

        # ── Right column: Analysis panels ──────────────────────────────────
        with gr.Column(scale=2):
            with gr.Group(elem_classes='panel'):
                gr.Markdown('### 🧠 Emotion Analysis')
                conf_plot = gr.Plot(label='Confidence Scores')

            with gr.Group(elem_classes='panel'):
                gr.Markdown('### 🔍 SHAP Token Highlights')
                gr.Markdown(
                    '<small>🟢 Green = pushes <b>toward</b> predicted emotion &nbsp;'
                    '🔴 Red = pushes <b>away</b></small>'
                )
                shap_display = gr.HTML(value='<p style="color:gray;">Send a message to see highlights.</p>')

    # ── Bottom row: Trend chart ────────────────────────────────────────────
    with gr.Row():
        with gr.Group(elem_classes='panel'):
            gr.Markdown('### 📈 Emotion Trend Across Conversation')
            trend_plot = gr.Plot(label='Emotion Trend')

    # ── Settings accordion ────────────────────────────────────────────────
    with gr.Accordion('⚙️ Generation Settings', open=False):
        with gr.Row():
            max_tokens_slider = gr.Slider(
                minimum=64, maximum=512, value=256, step=32,
                label='Max new tokens'
            )
            temperature_slider = gr.Slider(
                minimum=0.1, maximum=1.2, value=0.7, step=0.05,
                label='Temperature'
            )

    # ── Example inputs ────────────────────────────────────────────────────
    gr.Examples(
        examples=[
            ['I just got promoted today! I cannot believe it!'],
            ['I am so tired of being ignored by everyone around me.'],
            ['My grandmother passed away last week and I miss her so much.'],
            ['I am terrified about my exam results tomorrow.'],
            ['I love spending evenings with my family. They are my everything.'],
            ['Can you explain how machine learning works?'],
        ],
        inputs=msg_box,
        label='📝 Try these examples',
    )

    # ── Wire up events ────────────────────────────────────────────────────
    send_inputs  = [msg_box, chatbot, max_tokens_slider, temperature_slider, shap_toggle]
    send_outputs = [chatbot, conf_plot, shap_display, trend_plot]

    send_btn.click(
        fn=chat,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(lambda: '', outputs=msg_box)   # clear input after send

    msg_box.submit(
        fn=chat,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(lambda: '', outputs=msg_box)

    reset_btn.click(
        fn=reset_chat,
        outputs=send_outputs,
    )

print('Gradio app defined ✅')


In [ ]:
# ── Combined Gradio App Definition and Launch ────────────────────────────────
# This cell combines the logic from the original 'gradio_app' and 'launch' cells
# to ensure 'demo' is defined and launched in one step, especially after a kernel restart.

# ── State held outside Gradio (mutable across calls) ─────────────────────────
emotion_history = []   # list of (turn, label, confidence)

def chat(
    user_message: str,
    chat_history: list,
    max_tokens: int,
    temperature: float,
    run_shap: bool,
):
    """
    Main Gradio callback.
    Yields updates progressively so the UI feels responsive.
    """
    if not user_message.strip():
        yield chat_history, None, '<p>Please type a message.</p>', None
        return

    # Step 1: Emotion detection (fast)
    emo_result  = detect_emotion(user_message)
    emotion     = emo_result['label']
    confidence  = emo_result['confidence']
    scores      = emo_result['scores']
    color       = EMOTION_COLORS.get(emotion, '#888')

    # Update emotion history
    turn = len(emotion_history) + 1
    emotion_history.append((turn, emotion, confidence))

    conf_fig  = make_confidence_chart(scores)
    trend_fig = make_trend_chart(emotion_history)

    # Yield early so emotion info shows while Mistral generates
    yield (
        chat_history,
        conf_fig,
        f'<p style="color:{color};font-size:13px;">⏳ Generating response for <b>{emotion}</b> emotion...</p>',
        trend_fig,
    )

    # Step 2: Mistral response
    reply = mistral_reply(user_message, emotion, confidence, chat_history, max_tokens, temperature)

    # Update chat history
    chat_history = chat_history + [[user_message, reply]]

    # Step 3: SHAP (optional — slow ~15s per message)
    if run_shap:
        shap_html = get_shap_html(user_message, emotion)
    else:
        shap_html = (
            f'<div style="padding:10px;background:#f8f9fa;border-radius:8px;">'
            f'<b style="color:{color}">{emotion}</b> detected with '
            f'<b>{confidence:.1%}</b> confidence.<br>'
            f'<small style="color:gray;">Enable "Show SHAP" to see token highlights.</small></div>'
        )

    yield chat_history, conf_fig, shap_html, trend_fig


def reset_chat():
    global emotion_history
    emotion_history = []
    return [], None, '<p style="color:gray;">Conversation reset.</p>', make_trend_chart([])


# ── Gradio Layout ─────────────────────────────────────────────────────────────
with gr.Blocks(
    title='Emotion-Aware Chatbot',
    theme=gr.themes.Soft(primary_hue='blue', neutral_hue='slate'),
    css='''
        .chat-col   { min-height: 520px; }
        .panel      { background: #f8fafc; border-radius: 12px; padding: 12px; }
        footer      { display: none !important; }
    '''
) as demo:

    gr.Markdown(
        '# 🤖 Emotion-Aware Chatbot\n'
        '**RoBERTa** detects your emotion · **Mistral-7B** crafts an empathetic reply · '
        '**SHAP** shows which words drove the emotion'
    )

    with gr.Row():
        # ── Left column: Chat ──────────────────────────────────────────────
        with gr.Column(scale=3, elem_classes='chat-col'):
            chatbot = gr.Chatbot(
                label='Conversation',
                height=420,
                bubble_full_width=False,
                avatar_images=(None, '🤖'),
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder='Type your message here...',
                    show_label=False,
                    scale=5,
                    lines=2,
                )
                send_btn = gr.Button('Send 💬', variant='primary', scale=1)

            with gr.Row():
                reset_btn = gr.Button('🔄 Reset', variant='secondary', scale=1)
                shap_toggle = gr.Checkbox(
                    label='Show SHAP highlights (slower ~15s)',
                    value=False,
                    scale=2,
                )

        # ── Right column: Analysis panels ──────────────────────────────────
        with gr.Column(scale=2):
            with gr.Group(elem_classes='panel'):
                gr.Markdown('### 🧠 Emotion Analysis')
                conf_plot = gr.Plot(label='Confidence Scores')

            with gr.Group(elem_classes='panel'):
                gr.Markdown('### 🔍 SHAP Token Highlights')
                gr.Markdown(
                    '<small>🟢 Green = pushes <b>toward</b> predicted emotion &nbsp;'
                    '🔴 Red = pushes <b>away</b></small>'
                )
                shap_display = gr.HTML(value='<p style="color:gray;">Send a message to see highlights.</p>')

    # ── Bottom row: Trend chart ────────────────────────────────────────────
    with gr.Row():
        with gr.Group(elem_classes='panel'):
            gr.Markdown('### 📈 Emotion Trend Across Conversation')
            trend_plot = gr.Plot(label='Emotion Trend')

    # ── Settings accordion ────────────────────────────────────────────────
    with gr.Accordion('⚙️ Generation Settings', open=False):
        with gr.Row():
            max_tokens_slider = gr.Slider(
                minimum=64, maximum=512, value=256, step=32,
                label='Max new tokens'
            )
            temperature_slider = gr.Slider(
                minimum=0.1, maximum=1.2, value=0.7, step=0.05,
                label='Temperature'
            )

    # ── Example inputs ────────────────────────────────────────────────────
    gr.Examples(
        examples=[
            ['I just got promoted today! I cannot believe it!'],
            ['I am so tired of being ignored by everyone around me.'],
            ['My grandmother passed away last week and I miss her so much.'],
            ['I am terrified about my exam results tomorrow.'],
            ['I love spending evenings with my family. They are my everything.'],
            ['Can you explain how machine learning works?'],
        ],
        inputs=msg_box,
        label='📝 Try these examples',
    )

    # ── Wire up events ────────────────────────────────────────────────────
    send_inputs  = [msg_box, chatbot, max_tokens_slider, temperature_slider, shap_toggle]
    send_outputs = [chatbot, conf_plot, shap_display, trend_plot]

    send_btn.click(
        fn=chat,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(lambda: '', outputs=msg_box)   # clear input after send

    msg_box.submit(
        fn=chat,
        inputs=send_inputs,
        outputs=send_outputs,
    ).then(lambda: '', outputs=msg_box)

    reset_btn.click(
        fn=reset_chat,
        outputs=send_outputs,
    )

print('Gradio app defined ✅')

# share=True gives a public URL valid for 72h — perfect for thesis demo!
demo.launch(share=True, debug=False)


In [ ]:
# share=True gives a public URL valid for 72h — perfect for thesis demo!
demo.launch(share=True, debug=False)
